# Day 12 — Arithmetic, comparison and boolean logic

*100 Days of Polars · Phase 2: Expressions*

- **Post:** {{TODO: Medium link}}
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [1]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


Expressions overload Python's operators: `+ - * / // % **`, comparisons, and `& | ~` for boolean logic.
They run on whole columns at once, in Rust.

In [2]:
from datetime import datetime

items = pl.read_csv("data/raw/olist_order_items_dataset.csv", try_parse_dates=True)
orders = pl.read_csv("data/raw/olist_orders_dataset.csv", try_parse_dates=True)
payments = pl.read_csv("data/raw/olist_order_payments_dataset.csv")

## 1. Arithmetic

In [3]:
items.select(
    "price",
    "freight_value",
    total=pl.col("price") + pl.col("freight_value"),
    freight_pct=(pl.col("freight_value") / pl.col("price") * 100).round(1),
    price_k=pl.col("price") // 1000,
).head(5)

price,freight_value,total,freight_pct,price_k
f64,f64,f64,f64,f64
58.9,13.29,72.19,22.6,0.0
239.9,19.93,259.83,8.3,0.0
199.0,17.87,216.87,9.0,0.0
12.99,12.79,25.78,98.5,0.0
199.9,18.14,218.04,9.1,0.0


## 2. Comparison

In [4]:
items.select(
    "price",
    over_100=pl.col("price") > 100,
    over_100_method=pl.col("price").gt(100),  # same thing, method form
).head(5)

price,over_100,over_100_method
f64,bool,bool
58.9,false,false
239.9,true,true
199.0,true,true
12.99,false,false
199.9,true,true


In [5]:
# A boolean column sums to a count and averages to a share
items.select(
    n_over_100=(pl.col("price") > 100).sum(),
    share_over_100=(pl.col("price") > 100).mean().round(3),
)

n_over_100,share_over_100
u32,f64
40313,0.358


## 3. Combining conditions: `&`, `|`, `~`

In [6]:
cheap_but_heavy_freight = (pl.col("price") < 50) & (pl.col("freight_value") > 20)
items.filter(cheap_but_heavy_freight).height

4291

In [7]:
# Several arguments to filter are ANDed, no & needed
items.filter(pl.col("price") < 50, pl.col("freight_value") > 20).height

4291

In [8]:
items.filter(~cheap_but_heavy_freight).height

108359

**Gotcha: parentheses.** In Python, `&` binds tighter than `>`, and `a > b < c` is a *chained* comparison that calls `bool()`.

In [9]:
try:
    items.filter(pl.col("price") < 50 & pl.col("freight_value") > 20)
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

TypeError -> the truth value of an Expr is ambiguous


In [10]:
# `and` / `or` do not work either: they need a single True/False
try:
    items.filter((pl.col("price") < 50) and (pl.col("freight_value") > 20))
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

TypeError -> the truth value of an Expr is ambiguous


In [11]:
# Why `and` can't work row by row: it returns one side, it never combines them
print([True, False] and [False, False])   # the right-hand side
print([] and [False, False])              # the left-hand side (empty list is falsy)

[False, False]
[]


## 4. Ranges and sets: `is_between`, `is_in`

In [12]:
items.filter(pl.col("price").is_between(50, 100)).height  # closed="both" by default

33313

In [13]:
items.filter(pl.col("price").is_between(50, 100, closed="left")).height  # 50 <= price < 100

33141

In [14]:
# closed decides which ends are included
for closed in ["both", "left", "right", "none"]:
    n = items.filter(pl.col("price").is_between(50, 100, closed=closed)).height
    print(f"closed={closed!r:<8} -> {n:,} items")

closed='both'   -> 33,313 items
closed='left'   -> 33,141 items
closed='right'  -> 33,020 items
closed='none'   -> 32,848 items


In [15]:
# Black Friday week 2017 (Black Friday was 24 Nov 2017)
bf_week = pl.col("order_purchase_timestamp").is_between(datetime(2017, 11, 20), datetime(2017, 11, 27), closed="left")
orders.filter(bf_week).height

3008

In [16]:
# Gotcha: string bounds in is_between are read as COLUMN NAMES
try:
    orders.filter(pl.col("order_purchase_timestamp").is_between("2017-11-20", "2017-11-27"))
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

# Comparisons are different: the right-hand string is a value
print(orders.filter(pl.col("order_status") == "delivered").height)

ColumnNotFoundError -> unable to find column "2017-11-20"; valid columns: ["order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]
96478


**Dates in `is_between`: pass date/datetime objects, not strings.** Three ways, from simplest to most explicit. Watch the boundary: `datetime(2017, 11, 26)` means 26 Nov at 00:00:00.

In [17]:
from datetime import date

ts = pl.col("order_purchase_timestamp")

a = orders.filter(ts.dt.date().is_between(date(2017, 11, 20), date(2017, 11, 26)))                       # whole days
b = orders.filter(ts.is_between(datetime(2017, 11, 20), datetime(2017, 11, 27), closed="left"))           # next midnight, excluded
c = orders.filter(ts.is_between(pl.lit("2017-11-20").str.to_datetime(), pl.lit("2017-11-27").str.to_datetime(), closed="left"))  # parsed strings

print(a.height, b.height, c.height)  # all the same

3008 3008 3008


In [18]:
# The midnight trap: datetime(2017, 11, 26) is 00:00:00, so almost all of the 26th is lost
trap = orders.filter(ts.is_between(datetime(2017, 11, 20), datetime(2017, 11, 26)))
print(trap.height, "vs", a.height)

2617 vs 3008


In [19]:
payments.filter(pl.col("payment_type").is_in(["boleto", "voucher"])).group_by("payment_type").len()

payment_type,len
str,u32
"""boleto""",19784
"""voucher""",5775


## 5. Across columns: `any_horizontal`, `all_horizontal`

In [20]:
# Orders where any of the delivery timestamps is missing
orders.filter(
    pl.any_horizontal(pl.col("^order_delivered_.*_date$").is_null())
).group_by("order_status").len().sort("len", descending=True)

order_status,len
str,u32
"""shipped""",1107
"""canceled""",619
"""unavailable""",609
"""invoiced""",314
"""processing""",301
"""delivered""",9
"""created""",5
"""approved""",2


## 6. Nulls in comparisons

Comparing with a null gives **null**, not False, and `filter` drops null rows.

In [21]:
products = pl.read_csv("data/raw/olist_products_dataset.csv")
print("all rows:            ", products.height)
print("category == 'beleza_saude':", products.filter(pl.col("product_category_name") == "beleza_saude").height)
print("category != 'beleza_saude':", products.filter(pl.col("product_category_name") != "beleza_saude").height)
print("null category:       ", products.filter(pl.col("product_category_name").is_null()).height)

all rows:             32951
category == 'beleza_saude': 2444
category != 'beleza_saude': 29897
null category:        610


**Null means "unknown".** `2 == null` is not False, it is null. `&` and `|` only give a definite answer when the unknown cannot change it.

In [26]:
logic = pl.DataFrame({"x": [2, None, None], "flag": [True, False, True]}, schema={"x": pl.Int64, "flag": pl.Boolean})
logic.select(
    "x",
    "flag",
    x_eq_2=pl.col("x") == 2,                       # null where x is null
    x_eq_missing_null=pl.col("x").eq_missing(None),  # True where x is null
    null_and_flag=pl.lit(None, dtype=pl.Boolean) & pl.col("flag"),  # null & False = False, null & True = null
    null_or_flag=pl.lit(None, dtype=pl.Boolean) | pl.col("flag"),   # null | True = True, null | False = null
)

x,flag,x_eq_2,x_eq_missing_null,null_and_flag,null_or_flag
i64,bool,bool,bool,bool,bool
2,true,true,false,null,true
null,false,null,true,false,null
null,true,null,true,null,true


**Keeping the nulls when you mean "not X":** decide once in the filter. All three give the same count.

In [23]:
cat = pl.col("product_category_name")

print(products.filter(cat.ne_missing("beleza_saude")).height)               # null counts as "not equal"
print(products.filter((cat != "beleza_saude") | cat.is_null()).height)      # same, spelled out
print(products.filter(cat.fill_null("unknown") != "beleza_saude").height)   # fill first, then compare

30507
30507
30507


In [24]:
# pandas, for comparison: NaN != "x" is True, so missing rows are kept
products_pd = products.to_pandas()
print("pandas !=:", (products_pd["product_category_name"] != "beleza_saude").sum())
print("dtype:", products_pd["product_category_name"].dtype)

pandas !=: 30507
dtype: str


The `==` and `!=` counts do not add up to all rows: the null-category products fall into neither. That is SQL's three-valued logic.

## Try it yourself

1. Share of order items where freight costs more than the product itself.
2. Orders purchased on a weekend (`dt.weekday()` returns 6 and 7 for Saturday and Sunday).
3. Rewrite the `!=` filter above so null-category products are kept. (Hint: `ne_missing` or `| is_null()`.)